# Adsorption energies and free-energy diagram

## Why this matters

Adsorption energy is a descriptor that connects atomistic catalyst models to electrochemical activity of real catalysts. 
One of the ways to establish the connection is by using the Computational Hydrogen Electrode (CHE). 
In simulations, as a analogue to the Reversible Hydrogen Electrode (RHE), in measurements. 
CHE allows to convert adsorptionn energies into a potential-dependent free-energy diagram without explicitly adding solvated protons and electrons into simulations.

Let us take oxygen reduction reaction as an example.
You will (1) calculate O, OH, and OOH adsorption on Pt(111), (2) add the standard thermochemical corrections, and (3) construct an free-energy diagram at the equilibrium and limiting potentials.

## Learning objectives

By the end of this notebook, you should be able to:

- Build slab and adsorbate models.
- Set up and optimize a molecular FePc*OH model with controlled constraints.
- Calculate an OH adsorption free energy from consistent molecular references.
- Form O, OH, and OOH adsorption energies on Pt(111).
- Make the free-energy diagram.
- Calculate the limiting potential and thermodynamic overpotential.
- Compare database-derived metal free energies on one consistent CHE scale.

In [ ]:
# !pip install ase fairchem-core gdown

In [1]:
from ase import Atoms
from ase.build import fcc111, add_adsorbate, molecule
from ase.constraints import FixAtoms
from ase.db import connect
from ase.io import read
from ase.optimize import BFGS
from ase.visualize import view
from fairchem.core import FAIRChemCalculator, pretrained_mlip
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## Computational workflow

Computational electrocatalysis connects a scientific question to a reproducible chain of actions. Each step should be recorded so that the final trend can be traced back to the model and its assumptions. The workflow used in this workshop is illustrated below. We will be using FePc and Pt(111) as example catalysts.

![Computational electrocatalysis workflow](https://doublelayer.eu/vilab/wp-content/uploads/2026/08/image-1024x649.png)


In [ ]:
!gdown 1pw3jkPMWL6CLmpRGhBPdvzoLn4MzpTEX -O uma-s-1p1.pt

### Add the calculator

The UMA calculator is attached to every structure whose energy is evaluated. The model is distributed under the FAIR Chemistry License by Meta; see the [FAIR Chemistry License for UMA](https://huggingface.co/facebook/UMA/blob/main/LICENSE). By running this notebook you agree to its terms. For local use, obtain the model through the fairchem API or use the provided checkpoint without sharing personal access tokens.


In [2]:
calculator = FAIRChemCalculator(pretrained_mlip.load_predict_unit("uma-s-1p1.pt", device="cpu"), task_name="omat")

## Plotting conventions

Set shared figure dimensions, typography, and colors before producing the scientific plots.

In [3]:
plt.rc("figure", figsize=(8.25/2.54, 8.25/2.54))
plt.rc("figure", dpi=100)
plt.rc("lines",  linewidth=4)
plt.rc("font",   size=10)
plt.rc("font",   family="sans-serif")
plt.rc("axes",   titlesize=10)
plt.rc("axes",   labelsize=9)
plt.rc("xtick",  labelsize=8)
plt.rc("ytick",  labelsize=8)
plt.rc("legend", fontsize=8)
plt.rc("figure", titlesize=10)
colors = ["#1965B0", "#7BAFDE", "#90C987", "#F1932D", "#DC050C"]

## Oxygen reduction reaction

The four-electron oxygen reduction reaction and equilibrium potential ($U_\mathrm{eq}$ at STP) are

$$
\mathrm{O_2 + 4H^+ + 4e^- \longrightarrow 2H_2O},
\qquad U_{\mathrm{eq}}=1.23\ \mathrm{V}.
$$

We follow the associative pathway

$$
\mathrm{*+O_2 \rightarrow *OOH \rightarrow *O \rightarrow *OH
\rightarrow *+H_2O}.
$$

## Adsorption energy: minimal theory

Adsorption transfers a species from a reference state to a surface site. For
a molecular adsorbate A, the simplest reference reaction is

$$
\mathrm{A(g)+*\longrightarrow *A},
$$

where $*$ is a clean site and *A is the relaxed adsorbed state. The
adsorption energy is

$$
\Delta E_{\mathrm{ads}}
=E(\mathrm{*A})-E(*)-E(\mathrm{A}).
$$

With this convention, negative $\Delta E_{\mathrm{ads}}$ means that
adsorption is exothermic relative to the chosen references. 

Isolated O, OH, and OOH radicals are inconvenient electrochemical reference
states. We instead use balanced reactions involving H$_2$O and H$_2$. For
example,

$$
\mathrm{H_2O+*\longrightarrow *OH+\tfrac12H_2},
$$

gives

$$
\Delta E_{\mathrm{OH}}
=E(\mathrm{*OH})-E(*)-E(\mathrm{H_2O})
  +\tfrac12E(\mathrm{H_2}).
$$

It is often called an adsorption energy in electrocatalysis, so the reference reaction
should always be written explicitly. We first apply this idea to one molecular
example, FePc*OH, before introducing the more general Pt(111) workflow.

## Calculate molecular references

Before calculating adsorption energies, we need reference energies for H₂ and H₂O. These molecules appear in every thermodynamic cycle and chemical expression, so we optimize them first and store their energies.


In [4]:
H2 = molecule("H2")
H2.calc = calculator
optimizer = BFGS(H2, trajectory="H2.traj", logfile="H2.log")
optimizer.run(fmax=0.01)
energy_H2_ev = H2.get_potential_energy()
print(f"H2 reference energy: {energy_H2_ev:.2f} eV")

H2 reference energy: -6.64 eV


In [5]:
H2O = molecule("H2O")
H2O.calc = calculator
optimizer = BFGS(H2O, trajectory="H2O.traj", logfile="H2O.log")
optimizer.run(fmax=0.01)
energy_H2O_ev = H2O.get_potential_energy()
print(f"H2O reference energy: {energy_H2O_ev:.2f} eV")

H2O reference energy: -13.93 eV


## FePc*OH: a molecular adsorption example

Before defining reusable functions, let us set up one molecular catalyst by hand. The FePc structure is downloaded from the same test-model repository used in the electrocatalysis tutorials. The structure is available at [FePc.xyz](https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz). The phthalocyanine ring is held fixed while the central Fe atom and the added OH group are allowed to relax. This makes the `Atoms` object, calculator assignment, constraint mask, and optimization steps visible.

For this demonstration, the OH adsorption free energy uses the CHE reaction `H2O + FePc* -> FePc*OH + 1/2 H2`, with the same molecular corrections used later for Pt(111).

### Load FePc and optimize FePc

The FePc structure is a non-periodic molecular `Atoms` object. We relax only the central Fe atom while keeping the macrocycle fixed, so it provides a consistent FePc reference for the FePcOH calculation.


In [6]:
!wget -q https://gitlab.com/doublelayer/test-models/-/raw/main/iron-phthalocyanine-FePc-OH/FePc.xyz -O FePc.xyz
FePc = read("FePc.xyz")

In [7]:
FePc.calc = calculator
optimizer = BFGS(FePc, trajectory="FePc.traj", logfile="FePc.log")
optimizer.run(fmax=0.05)
energy_FePc_ev = FePc.get_potential_energy()
view(FePc, viewer="x3d")

### Add OH, constrain the ring, and optimize FePc*OH

The OH coordinates are placed above the Fe center. The constraint mask fixes every macrocycle atom and leaves only Fe and the two OH atoms movable.


In [8]:
FePcOH = FePc.copy()
FePcOH.set_constraint(FixAtoms(indices=[atom.index for atom in FePcOH if atom.symbol != 'Fe']))
add_adsorbate(FePcOH, Atoms("OH", positions=[[0, 0, 0], [0.0, 0.6, 0.6]]), height=1.8, position=FePcOH[-1].position[:2])
FePcOH.calc = calculator
optimizer = BFGS(FePcOH, trajectory="FePcOH.traj", logfile="FePcOH.log")
optimizer.run(fmax=0.05)
energy_FePcOH_ev = FePcOH.get_potential_energy()
view(FePcOH, viewer="x3d")

## Thermochemical corrections and CHE references

For each state we write

$$
G = E_{\mathrm{electronic}} + E_{\mathrm{ZPE}}
    + \int_0^T C_p\,dT - TS + \Delta G_{\mathrm{solv}} + \cdots
$$

The helper function above applies the same molecular thermal corrections
used in the course notebook and small tabulated corrections for the
adsorbates.  With H$_2$O and H$_2$ references, the three adsorption free
energies are

$$
\Delta G_{\mathrm{OH}}
=G(\mathrm{*OH})-G(*)-[G(\mathrm{H_2O})-\tfrac12G(\mathrm{H_2})],
$$

$$
\Delta G_{\mathrm{O}}
=G(\mathrm{*O})-G(*)-[G(\mathrm{H_2O})-G(\mathrm{H_2})],
$$

$$
\Delta G_{\mathrm{OOH}}
=G(\mathrm{*OOH})-G(*)-[2G(\mathrm{H_2O})-\tfrac32G(\mathrm{H_2})].
$$

Vibrational calculations, entropy models, solvation, and field corrections
can be refined independently, but they must be documented and applied with
the same convention to every catalyst being compared.


### Calculate the FePc OH adsorption free energy

The clean and adsorbed molecular energies are combined with the H2O/H2 CHE reference reaction. This is the same bookkeeping used in the Pt(111) helper below, but written out explicitly for one example.


In [9]:
energy_H2O_corrected_ev = energy_H2O_ev + 0.56 - 0.67
energy_H2_corrected_ev  = energy_H2_ev  + 0.27 - 0.41
free_energy_FePcOH_ev   = energy_FePcOH_ev - energy_FePc_ev - (energy_H2O_corrected_ev - 0.5 * energy_H2_corrected_ev) + 0.35 - 0.50
print(f"FePcOH adsorption free energy: {free_energy_FePcOH_ev:.3f} eV")

FePcOH adsorption free energy: 2.981 eV


## Build the reusable Pt(111) adsorption workflow

Let us define reusable functions because the clean slab, constraints, calculator, optimizer, and convergence criteria must remain identical across all intermediates.


In [ ]:
def relax_slab(metal="Pt", fmax=0.05, nx=2, ny=2, nz=3):
    atoms = fcc111(metal, (nx, ny, nz), vacuum=10.0)
    atoms.pbc = [True, True, True]
    atoms.set_constraint(FixAtoms(indices=[atom.index for atom in atoms if atom.index < nx * ny * (nz - 1)]))
    atoms.calc = calculator
    optimizer = BFGS(atoms, f"{metal}.traj", f"{metal}.log")
    optimizer.run(fmax=fmax)
    energy = atoms.get_potential_energy()
    return atoms, energy

In [ ]:
def relax_adsorbate(slab, adsorbate, position, metal="Pt", fmax=0.05, nx=2, ny=2, nz=3):
    atoms = slab.copy()
    if adsorbate == "O":
        ads = Atoms("O", positions=[(0, 0, 0)])
        height = 1.5
    if adsorbate == "OH":
        ads = Atoms("OH", positions=[(0, 0, 0), (0.7, 0, 0.7)])
        height = 1.8
    if adsorbate == "OOH":
        ads = Atoms("OOH", positions=[(0, 0, 0), (0.1, 0, 1.5), (0.7, 0, 2.2)])
        height = 1.8
    add_adsorbate(atoms, ads, height, position)
    atoms.pbc = [True, True, True]
    atoms.set_constraint(FixAtoms(indices=[atom.index for atom in atoms if atom.index < nx * ny * (nz - 1)]))
    atoms.calc = calculator
    optimizer = BFGS(atoms, f"{metal}{adsorbate}.traj", f"{metal}{adsorbate}.log")
    optimizer.run(fmax=fmax)
    energy = atoms.get_potential_energy()
    return atoms, energy


In [ ]:
def adsorption_energies(energy_Pt111_ev, energy_O_Pt111_ev, energy_OH_Pt111_ev, energy_OOH_Pt111_ev, energy_H2O_ev, energy_H2_ev):
    energy_H2O_ev = energy_H2O_ev + 0.56 - 0.67
    energy_H2_ev = energy_H2_ev + 0.27 - 0.41
    free_energy_O_ev = energy_O_Pt111_ev - energy_Pt111_ev - (energy_H2O_ev - energy_H2_ev) + 0.05 - 0
    free_energy_OH_ev = energy_OH_Pt111_ev - energy_Pt111_ev - (energy_H2O_ev - 0.5 * energy_H2_ev) + 0.35 - 0.50
    free_energy_OOH_ev = energy_OOH_Pt111_ev - energy_Pt111_ev - (2 * energy_H2O_ev - 1.5 * energy_H2_ev) + 0.40 - 0.30
    return free_energy_O_ev, free_energy_OH_ev, free_energy_OOH_ev

## Calculate adsorption energies

With the helper functions defined, we now optimize a Pt(111) slab and each slab+adsorbate combination (O, OH, OOH), storing the resulting energies in a pandas DataFrame.


In [ ]:
Pt111, energy_Pt111_ev = relax_slab()
O_Pt111, energy_O_Pt111_ev = relax_adsorbate(Pt111, "O", position="ontop")
OH_Pt111, energy_OH_Pt111_ev = relax_adsorbate(Pt111, "OH", position="ontop")
OOH_Pt111, energy_OOH_Pt111_ev = relax_adsorbate(Pt111, "OOH", position="ontop")

In [ ]:
view(Pt111, viewer="x3d")

In [ ]:
free_energy_O_ev, free_energy_OH_ev, free_energy_OOH_ev = adsorption_energies(energy_Pt111_ev, energy_O_Pt111_ev, energy_OH_Pt111_ev, energy_OOH_Pt111_ev, energy_H2O_ev, energy_H2_ev)
print(f"G_OOH = {free_energy_OOH_ev:.3f} eV")
print(f"G_O = {free_energy_O_ev:.3f} eV")
print(f"G_OH = {free_energy_OH_ev:.3f} eV")

## Construct the free-energy diagram

A free-energy diagram places every thermodynamic state on one common energy
scale. Each horizontal level is a state, not a transition state. The
dotted connectors only guide the eye; their height and shape do **not** give
an activation barrier or a reaction rate.

For neighboring states $i$ and $i+1$, the elementary reaction free energy is

$$
\Delta G_i(U)=G_{i+1}(U)-G_i(U).
$$

A negative value is downhill in the forward ORR direction. In the CHE, the
electrochemical potential of one proton-electron pair is replaced by

$$
\mu(\mathrm{H^+}+e^-)=\tfrac12G(\mathrm{H_2})-eU_{\mathrm{RHE}}.
$$

Consequently, changing the potential shifts a state by $-neU$ when $n$
proton-electron pairs separate that state from the chosen final reference.
When energies are reported in eV, the numerical shift is $-nU$ eV.

At potential $U$ versus RHE, the five free-energy levels are

$$
4.92-4U,\quad
\Delta G_{\mathrm{OOH}}-3U,\quad
\Delta G_{\mathrm{O}}-2U,\quad
\Delta G_{\mathrm{OH}}-U,\quad 0.
$$

The four downhill step energies at $U=0$ are

$$
4.92-\Delta G_{\mathrm{OOH}},\quad
\Delta G_{\mathrm{OOH}}-\Delta G_{\mathrm{O}},\quad
\Delta G_{\mathrm{O}}-\Delta G_{\mathrm{OH}},\quad
\Delta G_{\mathrm{OH}}.
$$

The code below reports the positive energy released by each reduction step
at $U=0$,

$$
s_i(0)=G_i(0)-G_{i+1}(0).
$$

For a one-electron step, $s_i(U)=s_i(0)-U$. The largest potential at which
every forward step remains downhill or thermoneutral is therefore

$$
U_\mathrm{L}=\min_i s_i(0).
$$

The thermodynamic overpotential is

$$
\eta_{\mathrm{TD}}=U_{\mathrm{eq}}-U_\mathrm{L}=1.23~\mathrm{V}-U_\mathrm{L}.
$$

At $U=1.23$ V the *overall* ORR free energy is zero, but individual steps
need not be thermoneutral. The step that first becomes uphill is the
potential-limiting step. This thermodynamic construction does not include
activation barriers, prefactors, double-layer effects, or mass transport;
it should not be identified directly with an experimental onset potential.

In [ ]:
def plot_at_potential(potential_v, intermediate_energies_ev, color="#0000cc", metal="Pt", peroxide=False):
    states = [4.92 - 4 * potential_v, intermediate_energies_ev[0] - 3 * potential_v, intermediate_energies_ev[1] - 2 * potential_v, intermediate_energies_ev[2] - potential_v, 0]
    if peroxide:
        plt.plot([1.2, 1.8], [states[1], 4.92 - 1.36 - 2 * potential_v], color="#DC050C", linestyle=":", linewidth=1, zorder=10)
        plt.plot([1.8, 2.2], [4.92 - 1.36 - 2 * potential_v, 4.92 - 1.36 - 2 * potential_v], color="#DC050C", linewidth=3, zorder=10)
    plt.plot([-0.2, 0.2], [states[0], states[0]], color=color, linewidth=3, label=f"{metal}, $U$={potential_v} V")
    plt.plot([0.8, 1.2], [states[1], states[1]], color=color, linewidth=3)
    plt.plot([1.8, 2.2], [states[2], states[2]], color=color, linewidth=3)
    plt.plot([2.8, 3.2], [states[3], states[3]], color=color, linewidth=3)
    plt.plot([3.8, 4.2], [states[4], states[4]], color=color, linewidth=3)
    plt.plot([0.2, 0.8], [states[0], states[1]], color=color, linestyle=":", linewidth=1)
    plt.plot([1.2, 1.8], [states[1], states[2]], color=color, linestyle=":", linewidth=1)
    plt.plot([2.2, 2.8], [states[2], states[3]], color=color, linestyle=":", linewidth=1)
    plt.plot([3.2, 3.8], [states[3], states[4]], color=color, linestyle=":", linewidth=1)


In [ ]:
def limiting_potential(intermediate_energies_ev):
    G_OOH, G_O, G_OH = intermediate_energies_ev
    step_energies_ev = np.array([4.92 - G_OOH, G_OOH - G_O, G_O - G_OH, G_OH])
    limiting_step = int(np.argmin(step_energies_ev))
    limiting_potential_v = float(step_energies_ev[limiting_step])
    overpotential_v = 1.23 - limiting_potential_v
    return step_energies_ev, limiting_step, limiting_potential_v, overpotential_v


In [ ]:
intermediate_energies_ev = [free_energy_OOH_ev, free_energy_O_ev, free_energy_OH_ev]
step_names = ["O2 to OOH*", "OOH* to O*", "O* to OH*", "OH* to H2O"]
step_energies_ev, limiting_step, limiting_potential_v, overpotential_v = limiting_potential(intermediate_energies_ev)
for name, energy_ev in zip(step_names, step_energies_ev):
    print(f"{name}: {energy_ev:.2f} eV")
print(f"Limiting step: {step_names[limiting_step]}")
print(f"U_L = {limiting_potential_v:.2f} V")
print(f"eta = {overpotential_v:.2f} V")


In [ ]:
plt.figure()
plot_at_potential(1.23, intermediate_energies_ev, colors[1], "Pt(111)")
plot_at_potential(round(limiting_potential_v, 2), intermediate_energies_ev, colors[2], "Pt(111)")
plot_at_potential(0.0, intermediate_energies_ev, colors[0], "Pt(111)")
plt.title(f"$\eta$ (Pt(111)) = {overpotential_v:.2f} V")
plt.xlabel("Reaction coordinate")
plt.ylabel("Free energies / eV")
plt.xticks([0, 1, 2, 3, 4], ["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
plt.legend()
plt.show()

## Database-derived metal data from Zenodo

The comparison below uses the BEEF-vdW molecular, clean-slab, and adsorbate databases from the [Zenodo record](https://zenodo.org/records/19023993), rather than hard-coded literature values. The same CHE reference reactions are applied to every metal.

The database accompanies T. K. Madsen, V. Ivanistsev, and J. Rossmeisl, *Getting Around the Intrinsic Errors of Density Functional Theory Calculations in Electrocatalysis*, **ACS Catal.** (2026), [doi:10.1021/acscatal.5c08550](https://doi.org/10.1021/acscatal.5c08550).

For FePc, the values G_OOH = 4.20 eV, G_O = 1.91 eV, and G_OH = 0.95 eV are taken from S. Favero et al., *Same FeN4 Active Site, Different Activity: How Redox Peaks Control Oxygen Reduction on Fe Macrocycles*, **ACS Electrochem.** 1 (2025) 617–632, [doi:10.1021/acselectrochem.4c00146](https://doi.org/10.1021/acselectrochem.4c00146).


In [ ]:
from pathlib import Path
from urllib.request import urlretrieve
import json

zenodo_base_url = "https://zenodo.org/records/19023993/files"
database_files = ["molecules.json", "slab_database.json", "adsorbate_database.json"]
for filename in database_files:
    database_path = Path(filename)
    if not database_path.exists():
        urlretrieve(f"{zenodo_base_url}/{filename}?download=1", database_path)

def load_json_records(filename):
    text = Path(filename).read_text(encoding="utf-16")
    decoder = json.JSONDecoder()
    records = []
    position = 0
    while position < len(text):
        while position < len(text) and text[position].isspace():
            position += 1
        if position >= len(text):
            break
        document, position = decoder.raw_decode(text, position)
        for key, value in document.items():
            if key.isdigit():
                records.append(value)
    return records

def records_for_functional(records):
    selected_records = []
    for row in records:
        key_value_pairs = row.get("key_value_pairs", {})
        if key_value_pairs.get("xc") == functional and row.get("energy") is not None:
            selected_records.append(row)
    return selected_records

functional = "BEEF-vdW"
molecule_records = load_json_records("molecules.json")
slab_records_json = load_json_records("slab_database.json")
adsorbate_records_json = load_json_records("adsorbate_database.json")
molecules = records_for_functional(molecule_records)

for row in molecules:
    if row["key_value_pairs"].get("smiles") == "O":
        energy_H2O_ev = row["energy"]
        break
for row in molecules:
    if row["key_value_pairs"].get("smiles") == "[HH]":
        energy_H2_ev = row["energy"]
        break
energy_H2O_ev = energy_H2O_ev + 0.56 - 0.67
energy_H2_ev = energy_H2_ev + 0.27 - 0.41

slab_records = []
for row in records_for_functional(slab_records_json):
    structure_string = row.get("key_value_pairs", {}).get("structure_str")
    if structure_string:
        metal = structure_string.split("_")[0]
        record = {"metal": metal, "energy_Pt111_ev": row["energy"]}
        slab_records.append(record)
slab_table = pd.DataFrame(slab_records)
slab_table = slab_table.drop_duplicates("metal")
slab_energies = slab_table.set_index("metal")

adsorbate_records = []
for row in records_for_functional(adsorbate_records_json):
    structure_string = row.get("key_value_pairs", {}).get("structure_str")
    if not structure_string:
        continue
    metal, unused, adsorbate, position = structure_string.split("_")
    if position == "top" and adsorbate in {"O", "OH", "OOH"}:
        record = {"metal": metal, "adsorbate": adsorbate, "energy_adsorbate_ev": row["energy"]}
        adsorbate_records.append(record)
adsorbate_table = pd.DataFrame(adsorbate_records)
adsorbate_table = adsorbate_table.drop_duplicates(["metal", "adsorbate"])
adsorbate_energies = adsorbate_table.pivot(index="metal", columns="adsorbate", values="energy_adsorbate_ev")
metal_energies = slab_energies.join(adsorbate_energies, how="inner")
metal_energies = metal_energies.dropna(subset=["O", "OH", "OOH"])
metal_energies["G_O"] = metal_energies["O"] - metal_energies["energy_Pt111_ev"] - (energy_H2O_ev - energy_H2_ev) + 0.05 - 0
metal_energies["G_OH"] = metal_energies["OH"] - metal_energies["energy_Pt111_ev"] - (energy_H2O_ev - 0.5 * energy_H2_ev) + 0.35 - 0.50
metal_energies["G_OOH"] = metal_energies["OOH"] - metal_energies["energy_Pt111_ev"] - (2 * energy_H2O_ev - 1.5 * energy_H2_ev) + 0.40 - 0.30
comparison_energies = metal_energies[["G_OOH", "G_O", "G_OH"]].copy()
comparison_energies.loc["FePc"] = {"G_OOH": 4.20, "G_O": 1.91, "G_OH": 0.95}
comparison_energies = comparison_energies.sort_index()
comparison_energies

In [ ]:
metal = "FePc"
metal_row = comparison_energies.loc[metal]
intermediate_energies_ev = [metal_row.G_OOH, metal_row.G_O, metal_row.G_OH]
step_energies_ev, limiting_step, limiting_potential_v, overpotential_v = limiting_potential(intermediate_energies_ev)
print(f"Catalyst: {metal}")
print(f"G_OOH = {intermediate_energies_ev[0]:.2f} eV")
print(f"G_O = {intermediate_energies_ev[1]:.2f} eV")
print(f"G_OH = {intermediate_energies_ev[2]:.2f} eV")
print(f"Limiting step: {step_names[limiting_step]}")
print(f"U_L = {limiting_potential_v:.2f} V")
print(f"eta = {overpotential_v:.2f} V")

In [ ]:
plt.figure()
plot_at_potential(1.23, intermediate_energies_ev, colors[1], metal)
plot_at_potential(round(limiting_potential_v, 2), intermediate_energies_ev, colors[2], metal)
plot_at_potential(0.0, intermediate_energies_ev, colors[0], metal)
plt.xlabel("Reaction coordinate")
plt.ylabel("Free energies / eV")
plt.xticks([0, 1, 2, 3, 4], ["*+O$_2$", "OOH*", "O*", "OH*", "*+H$_2$O"])
plt.legend()
plt.show()

## Summary

By working through this notebook, you have practiced how to:

- Build slab and adsorbate models.
- Set up and calculate the FePc*OH adsorption free energy.
- Form O, OH, and OOH adsorption energies.
- Make the free-energy diagram.
- Calculate the limiting potential and thermodynamic overpotential.
- Compare database-derived metal free energies on one consistent CHE scale.